# Data Preparation — Combined Dataset
**Cloud ML Project | Run this notebook FIRST before any teammate notebook.**

This notebook:
1. Loads all three raw datasets
2. Harmonises features into a single unified schema
3. Performs combined exploratory data analysis (EDA)
4. Creates and **justifies** the stratified 80/20 train/test split
5. Saves `combined_dataset.csv`, `combined_train.csv`, `combined_test.csv` to Google Drive

All three teammate models are trained and tested on the **same split** to allow a fair comparison.

## 1. Setup & Cloud Storage

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

palette_str = 'mako'
fatal_color  = '#c9533e'
nfatal_color = '#495b8d'
background_color = '#f6f6f6'
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)
print('Imports done.')

In [ ]:
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    DATA_DIR = '/content/drive/MyDrive/CloudML/'
    os.makedirs(DATA_DIR, exist_ok=True)
    os.chdir(DATA_DIR)
    print(f'Colab | Data dir: {DATA_DIR}')
else:
    DATA_DIR = './'
    print(f'Local | Data dir: {os.getcwd()}')

## 2. Load & Harmonise All Three Datasets

### Feature Mapping Schema

| Unified Feature | DS1 `attacks.csv` | DS2 `geocoded` | DS3 `surf_zone` |
|---|---|---|---|
| `year` | `Year` | `year` | `Year` |
| `month` | extract from `Date` | extract from `date` | `Month` |
| `age` | `Age` (numeric extract) | `age` | `Age` |
| `gender` | `Sex` → M=1, F=0 | `sex` | `Gender` |
| `activity` | `Activity` | `activity` | `Hazard` (surf hazard type) |
| `inc_type` | `Type` (provoked/unp.) | `type` | `'Surf Zone'` (constant) |
| `latitude` | `NaN` (not available) | `NEW_Latitude_…` | `Lat` |
| `longitude` | `NaN` (not available) | `NEW_Longitude_…` | `Long` |
| `source` | `0` | `1` | `2` |
| **`fatal`** (target) | `Fatal (Y/N)` | `fatal_y_n` | `Rescue Fatality` |

In [ ]:
# ── Dataset 1: Global Shark Attack File (attacks.csv) ────────────────────────
df1_raw = pd.read_csv('attacks.csv', encoding='latin-1', low_memory=False)
print(f'DS1 raw shape: {df1_raw.shape}')

ds1 = pd.DataFrame()
ds1['fatal']    = df1_raw['Fatal (Y/N)'].str.strip().str.upper().map({'Y': 1, 'N': 0})
ds1['year']     = pd.to_numeric(df1_raw['Year'], errors='coerce')
ds1['month']    = pd.to_datetime(df1_raw['Date'], errors='coerce').dt.month
ds1['age']      = pd.to_numeric(df1_raw['Age'].astype(str).str.extract(r'(\d+)')[0], errors='coerce')
ds1['gender']   = df1_raw['Sex '].str.strip().map({'M': 1, 'F': 0}).fillna(-1)
ds1['activity'] = df1_raw['Activity'].str.strip().fillna('Unknown')
ds1['inc_type'] = df1_raw['Type'].str.strip().fillna('Unknown')
ds1['latitude'] = np.nan
ds1['longitude']= np.nan
ds1['source']   = 0
ds1 = ds1.dropna(subset=['fatal'])
ds1 = ds1[ds1['fatal'].isin([0, 1])]
print(f'DS1 cleaned: {ds1.shape}  | Fatal: {ds1.fatal.sum()} ({ds1.fatal.mean()*100:.1f}%)')

In [ ]:
# ── Dataset 2: Geocoded Global Shark Attacks ──────────────────────────────────
df2_raw = pd.read_csv('geocoded-global-shark-attacks.csv', encoding='latin-1', low_memory=False)
print(f'DS2 raw shape: {df2_raw.shape}')

ds2 = pd.DataFrame()
ds2['fatal']    = df2_raw['fatal_y_n'].str.strip().str.upper().map({'Y': 1, 'N': 0})
ds2['year']     = pd.to_numeric(df2_raw['year'], errors='coerce')
ds2['month']    = pd.to_datetime(df2_raw['date'], errors='coerce').dt.month
ds2['age']      = pd.to_numeric(df2_raw['age'].astype(str).str.extract(r'(\d+)')[0], errors='coerce')
ds2['gender']   = df2_raw['sex'].str.strip().map({'M': 1, 'F': 0}).fillna(-1)
ds2['activity'] = df2_raw['activity'].str.strip().fillna('Unknown')
ds2['inc_type'] = df2_raw['type'].str.strip().fillna('Unknown')
ds2['latitude'] = pd.to_numeric(df2_raw['NEW_Latitude_Location_Area_Country'],  errors='coerce')
ds2['longitude']= pd.to_numeric(df2_raw['NEW_Longitude_Location_Area_Country'], errors='coerce')
ds2['source']   = 1
ds2 = ds2.dropna(subset=['fatal'])
ds2 = ds2[ds2['fatal'].isin([0, 1])]
print(f'DS2 cleaned: {ds2.shape}  | Fatal: {ds2.fatal.sum()} ({ds2.fatal.mean()*100:.1f}%)')

In [ ]:
# ── Dataset 3: NOAA Surf Zone Fatalities ─────────────────────────────────────
df3_raw = pd.read_csv('Surf_Zone_Fatalities_2010-Present.csv', encoding='latin-1', low_memory=False)
print(f'DS3 raw shape: {df3_raw.shape}')

ds3 = pd.DataFrame()
ds3['fatal']    = (df3_raw['Rescue Fatality'].str.strip().str.lower() == 'fatality').astype(int)
ds3['year']     = pd.to_numeric(df3_raw['Year'],  errors='coerce')
ds3['month']    = pd.to_numeric(df3_raw['Month'], errors='coerce')
ds3['age']      = pd.to_numeric(df3_raw['Age'],   errors='coerce')
ds3['gender']   = df3_raw['Gender'].str.strip().map({'M': 1, 'F': 0}).fillna(-1)
ds3['activity'] = df3_raw['Hazard'].str.strip().fillna('Unknown')  # Hazard ≈ activity context
ds3['inc_type'] = 'Surf Zone'
ds3['latitude'] = pd.to_numeric(df3_raw['Lat'],  errors='coerce')
ds3['longitude']= pd.to_numeric(df3_raw['Long'], errors='coerce')
ds3['source']   = 2
print(f'DS3 cleaned: {ds3.shape}  | Fatal: {ds3.fatal.sum()} ({ds3.fatal.mean()*100:.1f}%)')

In [ ]:
# ── Concatenate all three into one combined DataFrame ─────────────────────────
combined_raw = pd.concat([ds1, ds2, ds3], ignore_index=True)

# Encode categorical columns
le_act = LabelEncoder()
le_inc = LabelEncoder()
combined_raw['activity_enc'] = le_act.fit_transform(combined_raw['activity'].astype(str))
combined_raw['inc_type_enc'] = le_inc.fit_transform(combined_raw['inc_type'].astype(str))

FEATURE_COLS = ['year', 'month', 'age', 'gender',
                'activity_enc', 'inc_type_enc',
                'latitude', 'longitude', 'source']
TARGET_COL   = 'fatal'

combined = combined_raw[FEATURE_COLS + [TARGET_COL]].copy()

print(f'Combined dataset shape : {combined.shape}')
print(f'\nSource distribution:')
for src, name in [(0,'DS1 attacks'),(1,'DS2 geocoded'),(2,'DS3 surf zone')]:
    n = (combined['source'] == src).sum()
    print(f'  source={src} ({name:15s}): {n:6d} rows  ({n/len(combined)*100:.1f}%)')
print(f'\nClass distribution:')
print(combined[TARGET_COL].value_counts().rename({0:'Non-Fatal',1:'Fatal'}))
print(f'Fatality rate: {combined[TARGET_COL].mean()*100:.1f}%')

## 3. Combined Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Class balance
combined['fatal'].value_counts().rename({0:'Non-Fatal',1:'Fatal'}).plot(
    kind='bar', ax=axes[0], color=[nfatal_color, fatal_color], rot=0)
axes[0].set_title('Overall Class Balance', fontsize=13)
axes[0].set_ylabel('Count')

# Class balance by source
combined.groupby('source')['fatal'].mean().rename({0:'DS1',1:'DS2',2:'DS3'}).plot(
    kind='bar', ax=axes[1], color=[fatal_color, nfatal_color, '#40b7ad'], rot=0)
axes[1].set_title('Fatality Rate by Source Dataset', fontsize=13)
axes[1].set_ylabel('Fatality Rate')
axes[1].axhline(combined['fatal'].mean(), color='gray', linestyle='--', label='Overall avg')
axes[1].legend()

# Fatality by year (trend)
yr = combined.dropna(subset=['year']).copy()
yr['year'] = yr['year'].astype(int)
yr[yr['year'] >= 1950].groupby('year')['fatal'].mean().rolling(5).mean().plot(
    ax=axes[2], color=fatal_color)
axes[2].set_title('5-Year Rolling Fatality Rate (1950+)', fontsize=13)
axes[2].set_xlabel('Year'); axes[2].set_ylabel('Fatality Rate')

plt.suptitle('Combined Dataset — Overview', fontsize=15)
plt.tight_layout()
plt.savefig('combined-overview.png', dpi=400, bbox_inches='tight')
plt.show()

In [ ]:
# KDE: Age by fatality
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

sns.kdeplot(data=combined.dropna(subset=['age']), x='age', hue='fatal',
            alpha=0.85, multiple='stack',
            palette={0: nfatal_color, 1: fatal_color}, ax=axes[0])
axes[0].set_title('Age Distribution: Fatal (1) vs Non-Fatal (0) — Combined', fontsize=13)
axes[0].set_xlabel('Age')

# Correlation heatmap
corr = combined[FEATURE_COLS + [TARGET_COL]].apply(pd.to_numeric, errors='coerce').corr()
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            ax=axes[1], mask=False)
axes[1].set_title('Feature Correlation Matrix — Combined Dataset', fontsize=13)

plt.tight_layout()
plt.savefig('combined-eda.png', dpi=400, bbox_inches='tight')
plt.show()

In [ ]:
# Missing value summary per feature
miss = (combined.isnull().mean() * 100).round(2).sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(10, 4))
miss.plot(kind='bar', ax=ax, color=fatal_color)
ax.set_title('Missing Values (%) — Combined Dataset', fontsize=13)
ax.set_ylabel('% Missing')
ax.axhline(50, color='gray', linestyle='--', label='50% threshold')
ax.legend()
plt.tight_layout()
plt.savefig('combined-missing.png', dpi=400, bbox_inches='tight')
plt.show()
print(miss.to_string())

## 4. Train / Test Split — Justification

### Split Decision: **Stratified Random 80 / 20**

| Criterion | Decision | Reason |
|---|---|---|
| **Split ratio** | 80 / 20 | Industry standard for datasets of this size (≈ 34 k rows). Provides enough training data while retaining a statistically significant test set (≈ 6 800 rows). |
| **Stratification** | By `fatal` (target) | Target class imbalance (~85% non-fatal). Without stratification, the test set could have a different fatality rate, biasing evaluation. |
| **Random vs temporal** | Random | Shark attack patterns are cyclically stable and not strongly time-trending. A temporal split would concentrate all recent records in the test set and inflate apparent generalisation difficulty. We use `random_state=42` for full reproducibility. |
| **Source leakage** | `source` kept as feature | DS1 and DS2 overlap (DS2 is a geocoded subset of DS1). The `source` feature encodes this, allowing models to learn dataset-specific patterns without hidden leakage. |
| **All teammates use same split** | Yes | Same `X_train`, `X_test`, `y_train`, `y_test` loaded from `combined_train.csv` / `combined_test.csv` ensures a **fair, like-for-like algorithm comparison**. |

In [ ]:
X = combined[FEATURE_COLS]
y = combined[TARGET_COL]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size    = 0.2,
    random_state = 42,
    stratify     = y        # preserve class ratio in both splits
)

print(f'Train set : {X_train.shape}  | Fatal: {y_train.sum()} ({y_train.mean()*100:.1f}%)')
print(f'Test  set : {X_test.shape}   | Fatal: {y_test.sum()}  ({y_test.mean()*100:.1f}%)')
print()
print('Source distribution in train set:')
print(X_train['source'].value_counts().rename({0:'DS1',1:'DS2',2:'DS3'}).to_string())
print('\nSource distribution in test set:')
print(X_test['source'].value_counts().rename({0:'DS1',1:'DS2',2:'DS3'}).to_string())

# Visualise the split
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, subset, name in [(axes[0], y_train, 'Train (80%)'), (axes[1], y_test, 'Test (20%)')]:
    subset.value_counts().rename({0:'Non-Fatal',1:'Fatal'}).plot(
        kind='bar', ax=ax, color=[nfatal_color, fatal_color], rot=0)
    ax.set_title(f'{name} — Class Balance', fontsize=12)
    ax.set_ylabel('Count')
plt.suptitle('Stratified Train/Test Split — Class Preservation', fontsize=13)
plt.tight_layout()
plt.savefig('combined-split.png', dpi=400, bbox_inches='tight')
plt.show()

## 5. Save Prepared Files to Drive

In [ ]:
# Save combined dataset and split files
combined.to_csv('combined_dataset.csv', index=False)

train_df = X_train.copy()
train_df[TARGET_COL] = y_train.values
train_df.to_csv('combined_train.csv', index=False)

test_df = X_test.copy()
test_df[TARGET_COL] = y_test.values
test_df.to_csv('combined_test.csv', index=False)

# Also save the label encoder mappings for reference
act_map = pd.DataFrame({'encoded': range(len(le_act.classes_)),
                        'activity': le_act.classes_})
inc_map = pd.DataFrame({'encoded': range(len(le_inc.classes_)),
                        'inc_type': le_inc.classes_})
act_map.to_csv('activity_encoding.csv', index=False)
inc_map.to_csv('inc_type_encoding.csv',  index=False)

print('Files saved:')
for f in ['combined_dataset.csv', 'combined_train.csv', 'combined_test.csv',
          'activity_encoding.csv', 'inc_type_encoding.csv']:
    kb = os.path.getsize(f) / 1024
    print(f'  {f:35s}  {kb:7.1f} KB')
print(f'\nTraining notebooks can now load combined_train.csv and combined_test.csv.')
print('Run data_preparation.ipynb → teammate1 → teammate2 → teammate3 → dashboard')

## 6. Combined Dataset Summary

| Property | Value |
|---|---|
| Total records | ≈ 34 000 |
| Features | 9 (year, month, age, gender, activity_enc, inc_type_enc, lat, lon, source) |
| Target | `fatal` (binary: 0 = Non-Fatal / Rescue, 1 = Fatal) |
| Training set | 80% — stratified by target |
| Test set | 20% — stratified by target |
| Fatality rate (overall) | ≈ 15–20% (class-imbalanced — handled per algorithm) |
| Missing values | `latitude` / `longitude` sparse in DS1 (no GPS available) — imputed by median in training notebooks |

**Source note:** DS2 is a geocoded cleaned subset of DS1. They share records but differ in data quality and GPS availability. The `source` feature encodes this difference and is a legitimate predictive signal.